In [150]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [151]:
df=pd.read_csv('medical_insurance.csv')
print(df.head())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [152]:
df['sex']=df['sex'].map({
    'female':0,
    'male':1
})

In [153]:
df=df.drop(columns='region')
df['smoker']=df['smoker'].map({
    'yes':0,
    'no':1
})

In [154]:
df

,age,sex,bmi,children,smoker,charges
0,19,0,27.900,0,0,16884.92400
1,18,1,33.770,1,1,1725.55230
2,28,1,33.000,3,1,4449.46200
3,33,1,22.705,0,1,21984.47061
4,32,1,28.880,0,1,3866.85520
...,...,...,...,...,...,...
2767,47,0,45.320,1,1,8569.86180
2768,21,0,34.600,0,1,2020.17700
2769,19,1,26.030,1,0,16450.89470
2770,23,1,18.715,0,1,21595.38229


In [155]:
train=df.sample(frac=0.8,random_state=0)
test=df.drop(train.index)

x_train=train.drop(columns='charges')
y_train=train['charges']

x_test=test.drop(columns='charges')
y_test=test['charges']

x_train['intercept']=1
x_test['intercept']=1


In [156]:
print(x_train)
print(y_train)

      age  sex     bmi  children  smoker  intercept
897    19    1  25.555         1       1          1
2413   23    1  18.715         0       1          1
2576   31    0  32.775         2       1          1
723    19    1  35.400         0       1          1
619    55    0  37.100         0       1          1
...   ...  ...     ...       ...     ...        ...
288    59    0  36.765         1       0          1
2311   50    1  32.110         2       1          1
1181   24    0  29.925         0       1          1
337    62    1  27.550         1       1          1
46     18    0  38.665         2       1          1

[2218 rows x 6 columns]
897      2221.56445
2413    21595.38229
2576     5327.40025
723      1263.24900
619     10713.64400
           ...     
288     47896.79135
2311    25333.33284
1181     2850.68375
337     13937.66650
46       3393.35635
Name: charges, Length: 2218, dtype: float64


In [157]:
relation=train.corr()
relation

,age,sex,bmi,children,smoker,charges
age,1.000000,-0.024436,0.111825,0.043357,0.033009,0.300111
sex,-0.024436,1.000000,0.043443,0.027543,-0.083219,0.061331
bmi,0.111825,0.043443,1.000000,-0.011850,-0.019362,0.208645
children,0.043357,0.027543,-0.011850,1.000000,0.002078,0.054866
smoker,0.033009,-0.083219,-0.019362,0.002078,1.000000,-0.789450
charges,0.300111,0.061331,0.208645,0.054866,-0.789450,1.000000


In [158]:
import torch

In [159]:
X_numpy = x_train.to_numpy(dtype='float32')
y_numpy = y_train.to_numpy(dtype='float32')
x_tes=x_test.to_numpy(dtype='float32')
y_tes=y_test.to_numpy(dtype='float32')
x_torch=torch.from_numpy(X_numpy)
y_torch=torch.from_numpy(y_numpy).reshape(-1,1)
x_tes=torch.from_numpy(x_tes)
y_tes=torch.from_numpy(y_tes)

In [160]:
print(x_tes.size())
print(y_tes.size())
print(x_torch.size())
print(y_torch.size())

torch.Size([554, 6])
torch.Size([554])
torch.Size([2218, 6])
torch.Size([2218, 1])


In [161]:
xt=x_torch.t()
xt

tensor([[19.0000, 23.0000, 31.0000,  ..., 24.0000, 62.0000, 18.0000],
        [ 1.0000,  1.0000,  0.0000,  ...,  0.0000,  1.0000,  0.0000],
        [25.5550, 18.7150, 32.7750,  ..., 29.9250, 27.5500, 38.6650],
        [ 1.0000,  0.0000,  2.0000,  ...,  0.0000,  1.0000,  2.0000],
        [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
        [ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000]])

In [162]:
xt=torch.t(x_torch)
mul=torch.matmul(xt,x_torch)
inv=torch.inverse(mul)
fin=torch.matmul(inv,xt)
w=torch.matmul(fin,y_torch)

In [163]:
w
print(x_tes)

tensor([[19.0000,  0.0000, 27.9000,  0.0000,  0.0000,  1.0000],
        [33.0000,  1.0000, 22.7050,  0.0000,  1.0000,  1.0000],
        [37.0000,  0.0000, 27.7400,  3.0000,  1.0000,  1.0000],
        ...,
        [59.0000,  0.0000, 27.5000,  0.0000,  1.0000,  1.0000],
        [26.0000,  0.0000, 34.2000,  2.0000,  1.0000,  1.0000],
        [18.0000,  1.0000, 23.3200,  1.0000,  1.0000,  1.0000]])


In [164]:
y_pred=torch.matmul(x_tes,w)

In [165]:
val=(y_tes-y_pred)**2

In [166]:
mse=1/len(val)*(sum(val))

In [167]:
me=sum(mse)/len(mse)
me

tensor(2.6770e+08)

In [168]:
weights = torch.linalg.lstsq(x_torch, y_torch, driver='gelsd').solution

y_pred = x_tes @ weights

mse = torch.mean((y_tes - y_pred)**2)
rmse = torch.sqrt(mse)
rmse

tensor(16361.4639)

In [173]:
import torch.nn as nn
input_dim = x_torch.shape[1]
model = nn.Linear(input_dim, 1, bias=False)
optimizer = torch.optim.Adam(model.parameters(), lr=1)
criterion = nn.MSELoss()
epochs = 70000
for epoch in range(epochs):
    # Forward pass
    y_pred = model(x_torch)
    loss = criterion(y_pred, y_torch)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 500 == 0:
        current_rmse = torch.sqrt(loss).item()
        print(f'Epoch [{epoch+1}/{epochs}], RMSE: ${current_rmse:,.2f}')

model.eval()
with torch.no_grad():
    test_pred = model(x_tes)
    test_rmse = torch.sqrt(torch.mean((y_tes.reshape(-1,1) - test_pred)**2))
    print(f'\n--- Final Test RMSE: ${test_rmse.item():,.2f} ---')

Epoch [500/70000], RMSE: $11,325.12
Epoch [1000/70000], RMSE: $11,127.84
Epoch [1500/70000], RMSE: $10,946.31
Epoch [2000/70000], RMSE: $10,771.52
Epoch [2500/70000], RMSE: $10,600.89
Epoch [3000/70000], RMSE: $10,433.47
Epoch [3500/70000], RMSE: $10,268.89
Epoch [4000/70000], RMSE: $10,106.94
Epoch [4500/70000], RMSE: $9,947.53
Epoch [5000/70000], RMSE: $9,790.61
Epoch [5500/70000], RMSE: $9,636.21
Epoch [6000/70000], RMSE: $9,484.44
Epoch [6500/70000], RMSE: $9,335.42
Epoch [7000/70000], RMSE: $9,189.24
Epoch [7500/70000], RMSE: $9,045.97
Epoch [8000/70000], RMSE: $8,905.53
Epoch [8500/70000], RMSE: $8,767.75
Epoch [9000/70000], RMSE: $8,632.37
Epoch [9500/70000], RMSE: $8,499.06
Epoch [10000/70000], RMSE: $8,367.72
Epoch [10500/70000], RMSE: $8,238.38
Epoch [11000/70000], RMSE: $8,111.22
Epoch [11500/70000], RMSE: $7,986.45
Epoch [12000/70000], RMSE: $7,864.27
Epoch [12500/70000], RMSE: $7,744.84
Epoch [13000/70000], RMSE: $7,628.32
Epoch [13500/70000], RMSE: $7,514.83
Epoch [14000/